# Pipeline Evaluation

Today's drill is to build an evaluation harness from scratch to assess language generation systems. We will implement standard lexical overlap metrics (BLEU-1 and ROUGE-L) using pure Python and NumPy to evaluate machine-generated summaries against ground-truth references.

The pipeline will:
- Parse and tokenize text strings using basic tokenization rules.

- Implement BLEU-1 with modified unigram precision and a Brevity Penalty (BP).

- Implement ROUGE-L using a dynamic programming approach to find the Longest Common Subsequence (LCS).

- Aggregate and evaluate metrics across a sample evaluation dataset.

### Requirements

### Metric Formulation & Core Math

#### 1. BLEU-1 (Unigram Precision with Brevity Penalty)

* **Modified Unigram Precision**: For a candidate sequence, count occurrences of each unique token, but clip that count at the maximum frequency of that token in the corresponding reference text. This prevents "hacking" the metric via word repetition.
* **Brevity Penalty ($BP$)**: Penalizes candidate sentences that are shorter than the reference summary.

$$BP = \begin{cases} 1 & \text{if } c > r \\ e^{1 - \frac{r}{c}} & \text{if } c \le r \end{cases}$$

Where:
* $c$ is the candidate token length.
* $r$ is the reference token length.

* **Final BLEU-1 Score**: Calculated as the product of the brevity penalty and modified precision:

$$BLEU_1 = BP \times \text{Modified Precision}$$

#### 2. ROUGE-L (Longest Common Subsequence)

* **LCS via Dynamic Programming**: Identifies the longest sequence of words that appear in both texts in the same relative order (not necessarily consecutively). Given a reference sequence $R$ of length $m$ and a candidate sequence $C$ of length $n$, construct an $(m+1) \times (n+1)$ matrix $L$ where:

$$L[i][j] = \begin{cases} 0 & \text{if } i=0 \text{ or } j=0 \\ L[i-1][j-1] + 1 & \text{if } R[i-1] == C[j-1] \\ \max(L[i-1][j], L[i][j-1]) & \text{if } R[i-1] \neq C[j-1] \end{cases}$$

* **Evaluation Framework**:

* **LCS Precision**: 

$$P_{lcs} = \frac{LCS(R,C)}{|C|}$$

* **LCS Recall**: 

$$R_{lcs} = \frac{LCS(R,C)}{|R|}$$

* **ROUGE-L F1-Score**: 

$$ROUGE_L = \frac{2 \times P_{lcs} \times R_{lcs}}{P_{lcs} + R_{lcs}}$$

*(Note: If $P_{lcs} + R_{lcs} = 0$, then $ROUGE_L = 0$)*
```

### Expected Output
```
AUTOMATED PIPELINE EVALUATION HARNESS (v1)

Document Pair 1 
Reference: 'the administrator purged the sensitive social security numbers from the staging database yesterday'
Candidate: 'the administrator purged the social security numbers from the database yesterday'
BLEU-1:     0.9091
ROUGE-L P:  1.0000
ROUGE-L R:  0.7692
ROUGE-L F1: 0.8696

Document Pair 2 
Reference: 'call the manager back at the office or send an urgent email to administration'
Candidate: 'call the manager back at the office office office office'
BLEU-1:     0.4172
ROUGE-L P:  0.7778
ROUGE-L R:  0.5000
ROUGE-L F1: 0.6087


AGGREGATE METRICS SUMMARY
Mean BLEU-1:   0.6631
Mean ROUGE-L:  0.7391
```

### Imports

In [13]:
import math
import numpy as np
from collections import Counter

### Raw Validation Corpus

In [14]:
eval_dataset = [
    {
        "reference": "the administrator purged the sensitive social security numbers from the staging database yesterday",
        "candidate": "the administrator purged the social security numbers from the database yesterday"
    },
    {
        "reference": "call the manager back at the office or send an urgent email to administration",
        "candidate": "call the manager back at the office office office office"
    }
]

### Tokenization Pipeline

In [15]:
def tokenize(text):
    """ Applies basic lowercasing and splits strictly on whitespace. """
    return text.lower().strip().split()

### BLEU-1 Scoring Engine

In [16]:
def compute_bleu_1(reference_tokens, candidate_tokens):
    """ Calculates BLEU-1 score with modified unigram precision and brevity penalty. """
    c_len = len(candidate_tokens)
    r_len = len(reference_tokens)

    if c_len == 0:
        return 0.0
    
    # Modified Unigram Precision
    c_counts = Counter(candidate_tokens)
    r_counts = Counter(reference_tokens)

    clipped_hits = 0

    for token, count in c_counts.items():
        clipped_hits += min(count, r_counts.get(token, 0))

    precision = clipped_hits / c_len

    # Brevity Penalty Calculation
    if c_len > r_len:
        bp = 1.0
    else:
        bp = math.exp(1.0 -(r_len / c_len))

    return bp * precision

### ROUGE-L Scoring Engine

In [17]:
def compute_rouge_l(reference_tokens, candidate_tokens):
    """ Calculates ROUGE-L Precision, Recall, and F1 using Dynamic Programming. """
    r_len = len(reference_tokens)
    c_len = len(candidate_tokens)

    # Create LCS matrix
    lcs_matrix = np.zeros((r_len + 1, c_len + 1), dtype=int)

    for i in range(1, r_len + 1):
        for j in range(1, c_len + 1):
            if reference_tokens[i - 1] == candidate_tokens[j - 1]:
                lcs_matrix[i][j] = lcs_matrix[i - 1][j - 1] + 1
            else:
                lcs_matrix[i][j] = max(lcs_matrix[i - 1][j], lcs_matrix[i][j - 1])

    lcs_count = lcs_matrix[r_len][c_len]

    # Score Metrics
    precision = lcs_count / c_len if c_len > 0 else 0.0
    recall = lcs_count / r_len if r_len > 0 else 0.0

    if precision + recall == 0:
        f1_score = 0.0
    else:
        f1_score = (2 * precision * recall) / (precision + recall)

    return {"precision":precision, "recall": recall, "f1": f1_score}

### Execution and Evaluation Harness

In [18]:
def evaluate_pipeline(dataset):
    """ Iterates through the dataset, runs metrics, and logs performance details. """
    results = []

    print("AUTOMATED PIPELINE EVALUATION HARNESS (v1)\n")

    for idx, item in enumerate(dataset, 1):
        ref_token = tokenize(item["reference"])
        cand_token = tokenize(item["candidate"])

        bleu1 = compute_bleu_1(ref_token, cand_token)
        rouge_metrics = compute_rouge_l(ref_token, cand_token)

        results.append({
            "bleu1": bleu1,
            "rouge_precision": rouge_metrics["precision"],
            "rouge_recall": rouge_metrics["recall"],
            "rouge_f1": rouge_metrics["f1"]
        })

        print(f"Document Pair {idx}")
        print(f"Reference: '{item['reference']}'")
        print(f"Candidate: '{item['candidate']}'")
        print(f"BLEU-1:     {bleu1:.4f}")
        print(f"ROUGE-L P:  {rouge_metrics['precision']:.4f}")
        print(f"ROUGE-L R:  {rouge_metrics['recall']:.4f}")
        print(f"ROUGE-L F1: {rouge_metrics['f1']:.4f}\n")

    # Aggregate Metrics
    avg_bleu1 = np.mean([res["bleu1"] for res in results])
    avg_rouge_f1 = np.mean([res["rouge_f1"] for res in results])

    print("AGGREGATE METRICS SUMMARY")
    print(f"Mean BLEU-1: {avg_bleu1:.4f}")
    print(f"Mean ROUGE-L: {avg_rouge_f1:.4f}")

### Execute Pipeline Evaluation

In [19]:
evaluate_pipeline(eval_dataset)

AUTOMATED PIPELINE EVALUATION HARNESS (v1)

Document Pair 1
Reference: 'the administrator purged the sensitive social security numbers from the staging database yesterday'
Candidate: 'the administrator purged the social security numbers from the database yesterday'
BLEU-1:     0.8338
ROUGE-L P:  1.0000
ROUGE-L R:  0.8462
ROUGE-L F1: 0.9167

Document Pair 2
Reference: 'call the manager back at the office or send an urgent email to administration'
Candidate: 'call the manager back at the office office office office'
BLEU-1:     0.4692
ROUGE-L P:  0.7000
ROUGE-L R:  0.5000
ROUGE-L F1: 0.5833

AGGREGATE METRICS SUMMARY
Mean BLEU-1: 0.6515
Mean ROUGE-L: 0.7500
